# Circles in Ellipses
n unit circles (radius 1) packed inside an ellipse of minimum area. Both semi-axes (a, b) are free --
the ellipse's shape is optimized along with the circle positions, not fixed in advance. Run all cells
top to bottom.

Per n: random restarts, warm start from n-1 (one more circle inserted), basin hopping, then a 60-digit
validity certificate (proves no overlap and full containment -- not proof of optimality), then SVG + JSON.
Resumes if interrupted in the same runtime (finished n are skipped). Extend `NS` below to widen the range.

Sanity-checked against the known n=1 case (a=b=1) and multi-seed-verified for n=2 (converges to the exact
closed form area = 3sqrt(3)/2 from every random seed tried) and n=3 (5 of 6 seeds agree) -- see chat for detail.

**Increasing accuracy:** raise `SEARCH_SEC` / `HOP_SEC` in the loop cell if area is still dropping noticeably
near the end of a phase; if it has flattened, more time won't help -- a different seed will. If a value looks
suspiciously worse than its neighbors, rerun with more seeds -- n=3 above shows the search can occasionally
land in a worse local optimum.

In [ ]:
!pip -q install jax jaxlib scipy mpmath numpy

In [ ]:
import json
# n unit circles packed in an ellipse (semi-axes a,b both free), minimizing area a*b.
import json, time
import numpy as np
import jax, jax.numpy as jnp
from scipy.optimize import minimize
import mpmath as mp

jax.config.update("jax_enable_x64", True)
mp.mp.dps = 60
RC = 1.0   # unit circle radius


_GRID_N = 256
_GRID_TH = jnp.linspace(0, 2*jnp.pi, _GRID_N, endpoint=False)


def _nearest_on_ellipse(x0, y0, a, b, newton_iters=20):
    """Nearest point on ellipse (x/a)^2+(y/b)^2=1 to (x0,y0): coarse grid for the
    right basin (avoids saddle/wrong-critical-point traps), then Newton polish.
    Fully differentiable (grid pick is a constant w.r.t. the Newton refinement)."""
    c, s = jnp.cos(_GRID_TH), jnp.sin(_GRID_TH)
    d2 = (a*c - x0)**2 + (b*s - y0)**2
    th = _GRID_TH[jnp.argmin(d2)]
    for _ in range(newton_iters):
        c, s = jnp.cos(th), jnp.sin(th)
        f = a*s*(a*c - x0) - b*c*(b*s - y0)
        fp = (a*a - b*b)*(c*c - s*s) - a*x0*c - b*y0*s
        th = th - f / jnp.where(jnp.abs(fp) < 1e-12, 1e-12, fp)
    c, s = jnp.cos(th), jnp.sin(th)
    return a*c, b*s


def _ellipse_boundary_dist(x0, y0, a, b):
    """Signed distance from (x0,y0) to the boundary of ellipse x^2/a^2+y^2/b^2=1.
    Positive = point is inside."""
    px, py = _nearest_on_ellipse(x0, y0, a, b)
    dist = jnp.sqrt((x0-px)**2 + (y0-py)**2)
    inside = (x0/a)**2 + (y0/b)**2 <= 1.0
    return jnp.where(inside, dist, -dist)


def unpack(z, n):
    return z[:n], z[n:2*n], z[2*n], z[2*n+1]     # x, y, a, b


def cont_viol(z, n):
    x, y, a, b = unpack(z, n)
    d = jax.vmap(lambda xi, yi: _ellipse_boundary_dist(xi, yi, a, b))(x, y)
    return RC - d          # <=0 feasible (circle fits with margin)


def pair_sep(z, n):
    x, y, a, b = unpack(z, n)
    dx = x[:, None] - x[None, :]; dy = y[:, None] - y[None, :]
    dist = jnp.sqrt(dx**2 + dy**2 + 1e-18)
    return dist - 2 * RC


def energy(z, n):
    cv = jnp.maximum(cont_viol(z, n), 0.0)
    ms = pair_sep(z, n)
    iu = jnp.triu(jnp.ones((n, n)), k=1)
    pen = jnp.maximum(-ms, 0.0) * iu
    return jnp.sum(cv**2) + jnp.sum(pen**2)


def make_fns(n):
    def obj(z, mu):
        a, b = z[2*n], z[2*n+1]
        return a * b + mu * energy(z, n)
    vg = jax.jit(jax.value_and_grad(obj))
    CV = jax.jit(lambda z: cont_viol(z, n))
    PS = jax.jit(lambda z: pair_sep(z, n))
    return vg, CV, PS


def max_violation(z, n, CV, PS):
    cv = float(jnp.max(CV(z)))
    if n < 2: return cv
    ms = np.array(PS(z)); iu = np.triu_indices(n, 1)
    return max(cv, float(-ms[iu].min()))


def penalty_solve(z0, n, vg, mus=(1e1,1e2,1e3,1e4,1e5,1e6), maxiter=400):
    z = z0.copy()
    for mu in mus:
        f = lambda zz: tuple(np.asarray(a_, dtype=np.float64) for a_ in vg(zz, mu))
        r = minimize(f, z, jac=True, method="L-BFGS-B", options={"maxiter": maxiter, "maxcor": 30})
        z = r.x
    return z


def feasible_scale(z, n, CV, PS):
    """dilate positions AND ellipse together by k (circle radius stays fixed at 1, so this
    both grows the container and spreads circle centres apart, resolving both containment
    and pairwise overlap -- same scaling trick as the polygon engine)."""
    x, y, a, b = unpack(jnp.asarray(z), n)
    def viol(k):
        zz = jnp.concatenate([x*k, y*k, jnp.array([a*k, b*k])])
        return max_violation(zz, n, CV, PS)
    if viol(1.0) <= 0: return np.array(z)
    hi = 1.0 + 1e-6
    while viol(hi) > 0:
        hi = 1.0 + (hi-1.0)*2
        if hi > 4: return None
    lo = 1.0
    for _ in range(45):
        mid = 0.5*(lo+hi)
        if viol(mid) > 0: lo = mid
        else: hi = mid
    k = hi
    return np.array(jnp.concatenate([x*k, y*k, jnp.array([a*k, b*k])]))


def slsqp_polish(z, n, CV, PS, iters=250):
    z = np.array(z)
    def cons(zz):
        zz = jnp.asarray(zz)
        cv = -cont_viol(zz, n)
        ps = pair_sep(zz, n)
        iu = np.triu_indices(n, 1)
        return jnp.concatenate([cv, ps[iu]]) if n > 1 else cv
    cj = jax.jit(cons); cjac = jax.jit(jax.jacfwd(cons))
    def obj(zz): return zz[2*n]*zz[2*n+1]
    def objg(zz):
        g = np.zeros(2*n+2); g[2*n] = zz[2*n+1]; g[2*n+1] = zz[2*n]
        return g
    Rbig = 3*(z[2*n] + z[2*n+1]) + 10
    bounds = [(-Rbig, Rbig)]*(2*n) + [(1e-6, Rbig), (1e-6, Rbig)]
    res = minimize(obj, z, jac=objg, bounds=bounds,
                   constraints=[{"type": "ineq", "fun": lambda zz: np.asarray(cj(zz)),
                                 "jac": lambda zz: np.asarray(cjac(zz))}],
                   method="SLSQP", options={"maxiter": iters, "ftol": 1e-13})
    return res.x


def s_estimate(n, density=0.75):
    # area ~ n*pi*RC^2/density ; start near-circular (a=b)
    area = n*np.pi*RC*RC/density
    r = np.sqrt(area/np.pi)
    return r, r


def random_init(n, a0, b0, rng):
    pts = []
    while len(pts) < n:
        ang = rng.random()*2*np.pi; u = rng.random()
        rad = np.sqrt(u)
        pts.append((a0*rad*0.85*np.cos(ang), b0*rad*0.85*np.sin(ang)))
    pts = np.array(pts)
    return np.concatenate([pts[:,0], pts[:,1], [a0, b0]])


def run(n, seconds, seed=0, verbose=True, out=None):
    rng = np.random.default_rng(seed)
    vg, CV, PS = make_fns(n)
    best = None; t0=time.time(); trial=0
    while time.time()-t0 < seconds:
        trial += 1
        a0, b0 = s_estimate(n)
        a0 *= (1.15+0.3*rng.random()); b0 *= (1.15+0.3*rng.random())
        z0 = random_init(n, a0, b0, rng)
        z = penalty_solve(z0, n, vg)
        zf = feasible_scale(z, n, CV, PS)
        if zf is None: continue
        area = zf[2*n]*zf[2*n+1]
        if best is not None and area > best[2*n]*best[2*n+1] + 0.5: continue
        try:
            zp = slsqp_polish(zf, n, CV, PS)
            zp2 = feasible_scale(zp, n, CV, PS)
            if zp2 is not None and zp2[2*n]*zp2[2*n+1] < area: zf = zp2; area = zf[2*n]*zf[2*n+1]
        except Exception: pass
        if best is None or area < best[2*n]*best[2*n+1]:
            best = zf
            if verbose: print(f"n={n} trial={trial} t={time.time()-t0:.0f}s area={area:.6f} a={zf[2*n]:.4f} b={zf[2*n+1]:.4f}", flush=True)
            if out: json.dump({"n": n, "area": float(area), "z": best.tolist()}, open(out, "w"))
    return best


def hop(n, seconds, seed, start_z, out=None):
    rng = np.random.default_rng(seed)
    vg, CV, PS = make_fns(n)
    best = np.array(start_z); cur = best.copy(); t0=time.time(); it=0
    def area_of(zz): return zz[2*n]*zz[2*n+1]
    while time.time()-t0 < seconds:
        it += 1
        zz = cur.copy()
        k = rng.integers(1, max(2, n//4+1))
        idx = rng.choice(n, k, replace=False)
        sig = rng.choice([0.15, 0.35, 0.6])
        zz[idx] += rng.normal(0, sig, k); zz[n+idx] += rng.normal(0, sig, k)
        if rng.integers(3) == 0:
            j = rng.integers(n); a, b = zz[2*n], zz[2*n+1]
            ang = rng.random()*2*np.pi; rad = np.sqrt(rng.random())*0.85
            zz[j] = a*rad*np.cos(ang); zz[n+j] = b*rad*np.sin(ang)
        zz[2*n] *= 1.03; zz[2*n+1] *= 1.03
        z1 = penalty_solve(zz, n, vg, mus=(1e2,1e3,1e4,1e5,1e6), maxiter=300)
        zf = feasible_scale(z1, n, CV, PS)
        if zf is None or area_of(zf) > area_of(best)+0.05: continue
        try:
            zp = slsqp_polish(zf, n, CV, PS)
            z2 = feasible_scale(zp, n, CV, PS)
            if z2 is not None and area_of(z2) < area_of(zf): zf = z2
        except Exception: pass
        if area_of(zf) < area_of(cur)+0.02: cur = zf
        if area_of(zf) < area_of(best)-1e-9:
            best = zf; print(f"n={n} hop it={it} t={time.time()-t0:.0f}s area={area_of(zf):.6f}", flush=True)
            if out: json.dump({"n": n, "area": float(area_of(zf)), "z": best.tolist()}, open(out, "w"))
    return best


def grow(zprev, n, rng, tries=6):
    m = n-1
    vg, CV, PS = make_fns(n); best=None
    x,y,a,b = zprev[:m], zprev[m:2*m], zprev[2*m], zprev[2*m+1]
    for _ in range(tries):
        ang=rng.random()*2*np.pi; rad=np.sqrt(rng.random())*0.85
        z0 = np.concatenate([x,[a*rad*np.cos(ang)], y,[b*rad*np.sin(ang)], [a*1.04, b*1.04]])
        zf = feasible_scale(penalty_solve(z0,n,vg,mus=(1e2,1e3,1e4,1e5,1e6),maxiter=300), n, CV, PS)
        if zf is not None and (best is None or zf[2*n]*zf[2*n+1] < best[2*n]*best[2*n+1]): best=zf
    return best


def certify(z, n, eps=1e-9):
    """60-digit proof: every circle strictly inside the ellipse, every pair strictly separated."""
    z = [mp.mpf(float(v)) for v in z]
    x = z[:n]; y = z[n:2*n]; a = z[2*n]; b = z[2*n+1]
    k = 1 - mp.mpf(eps)          # shrink circles slightly (safety margin), NOT the ellipse
    r = RC * (1 - mp.mpf(eps))
    a2 = a * (1 + mp.mpf(eps)); b2 = b * (1 + mp.mpf(eps))   # grow ellipse slightly (safety margin)

    def boundary_dist(x0, y0):
        best_th, best_d2 = None, mp.inf
        for k in range(256):                     # coarse grid: right basin
            th = 2*mp.pi*k/256
            c, s = mp.cos(th), mp.sin(th)
            d2 = (a2*c-x0)**2 + (b2*s-y0)**2
            if d2 < best_d2: best_d2, best_th = d2, th
        th = best_th
        for _ in range(80):                       # Newton polish to 60 digits
            c, s = mp.cos(th), mp.sin(th)
            f = a2*s*(a2*c-x0) - b2*c*(b2*s-y0)
            fp = (a2*a2-b2*b2)*(c*c-s*s) - a2*x0*c - b2*y0*s
            if fp == 0: break
            th = th - f/fp
        c, s = mp.cos(th), mp.sin(th)
        px, py = a2*c, b2*s
        d = mp.sqrt((x0-px)**2 + (y0-py)**2)
        inside = (x0/a2)**2 + (y0/b2)**2 <= 1
        return d if inside else -d

    cont = mp.inf
    for i in range(n):
        cont = min(cont, boundary_dist(x[i], y[i]) - r)
    pair = mp.inf
    for i in range(n):
        for j in range(i+1, n):
            d = mp.hypot(x[i]-x[j], y[i]-y[j])
            pair = min(pair, d - 2*r)
    assert cont > 0 and (n < 2 or pair > 0), (float(cont), float(pair))
    return a, b, cont, pair


def page_value(v): return f"{mp.floor(v*10**5)/10**5}+"


def geometry(z, n, W=280):
    x, y, a, b = z[:n], z[n:2*n], float(z[2*n]), float(z[2*n+1])
    pad = 2; sc = (W-2*pad)/(2*a); H = 2*b*sc+2*pad
    cx, cy = W/2, H/2
    P = lambda px, py: (cx+px*sc, cy-py*sc)
    circles = [(P(x[i], y[i]), RC*sc) for i in range(n)]
    ell = [P(a*np.cos(t), b*np.sin(t)) for t in np.linspace(0, 2*np.pi, 120)]
    return circles, ell, W, H


def write_svg(z, n, path):
    circles, ell, W, H = geometry(z, n)
    body = "".join(f'<circle cx="{cx:.3f}" cy="{cy:.3f}" r="{r:.3f}" fill="#ffffe7" stroke="#000" stroke-width="0.75"/>'
                   for (cx, cy), r in circles)
    pts = " ".join(f"{x:.3f},{y:.3f}" for x, y in ell)
    open(path, "w").write(f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H:.3f}" viewBox="0 0 {W} {H:.3f}">'
                          f'{body}<polygon points="{pts}" fill="none" stroke="#000" stroke-width="1.5"/></svg>')


In [ ]:
NS = range(25, 51)
SEARCH_SEC = 240   # random restarts per n; raise if area is still dropping near the end of this phase
HOP_SEC = 120        # basin hopping per n; raise for higher n
OUT = "/content/circles_in_ellipses"; import os; os.makedirs(OUT, exist_ok=True)

prev, rows = None, []
for n in NS:
    jf = f"{OUT}/{n}.json"
    if os.path.exists(jf):
        d = json.load(open(jf)); prev = __import__("numpy").array(d["z"]); rows.append((n, d["a"], d["b"], d["area"])); continue
    rng = __import__("numpy").random.default_rng(n)
    cands = [c for c in (run(n, SEARCH_SEC, n, verbose=False), grow(prev, n, rng) if prev is not None else None) if c is not None]
    if not cands:
        print(f"n={n}  no feasible layout found in budget -- skipping (rerun this cell later to retry)", flush=True)
        continue
    start = min(cands, key=lambda c: c[2*n]*c[2*n+1])
    best = hop(n, HOP_SEC, n + 1000, start, "/tmp/hop.json")
    a, b, cs, pg = certify(best, n)
    area = float(a*b)
    write_svg(best, n, f"{OUT}/circles_in_ellipses_n{n}.svg")
    json.dump({"n": n, "a": float(a), "b": float(b), "area": area, "z": [float(v) for v in best]}, open(jf, "w"))
    rows.append((n, float(a), float(b), area))
    prev = best
    print(f"n={n}  certified a={mp.nstr(a,10)} b={mp.nstr(b,10)}  a+ b+ = {page_value(a)} {page_value(b)}  area={area:.6f}", flush=True)

import csv
with open(f"{OUT}/summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["n", "a", "b", "area"])
    for row in rows: w.writerow(row)
for (na, aa, ba, areaa), (nb, ab, bb, areab) in zip(rows, rows[1:]):
    if areab < areaa: print(f"warning: area({nb}) < area({na}); n={na} search is weak, rerun it longer")
import shutil
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files; files.download(OUT + ".zip")
except ImportError: pass